# SwiftRoute Logistics — SQL Analysis 

**Project stage 2 of 4:** Excel (data prep/QA) → **SQLite / SQL (this notebook)** → Python (statistics & ML) → Power BI (executive dashboard)

This notebook represents the work a data/business analyst would normally do directly against a company data warehouse: loading the cleaned extracts into a relational database and using SQL — joins, aggregations, window functions, CTEs — to answer operational business questions that need to be queried on demand, not just visualized.

**Database:** `swiftroute.db` (SQLite), built from the four cleaned CSV extracts: `orders` (27,979 rows), `drivers` (55), `hubs` (6), `vehicles` (45).

## Contents

1. Schema Reference
2. Business Question 1 — What are the top reasons orders get delayed, and does the leading cause differ by hub?
3. Business Question 2 — What is the order cancellation rate, and does it vary meaningfully by hub, vehicle type, or day of week?
4. Business Question 3 — Which hubs are running over capacity, and how has that changed month by month?
5. Business Question 4 — How evenly is delivery workload distributed across drivers?
6. Business Question 5 — Which vehicles are under-utilized or completely idle, and which are overworked?
7. Business Question 6 — Does employment type (full-time / part-time / contract) relate to delivery performance?
8. Business Question 7 — Does driver tenure (time since hire) relate to delay rate?
9. Business Question 8 — Validate the dashboard's month-over-month order growth figures directly from SQL
10. Summary of SQL-layer findings

In [1]:
import sqlite3
import pandas as pd
pd.set_option('display.max_rows', 20)

conn = sqlite3.connect('swiftroute.db')

for t in ['orders', 'drivers', 'hubs', 'vehicles']:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows")

orders: 27979 rows
drivers: 55 rows
hubs: 6 rows
vehicles: 45 rows


## Schema Reference

Quick look at each table so queries below are easy to follow.

> **Data quality note (carried over from the Excel QA workbook):** the `Driver_Name` text stored inside `orders` does **not** reliably match the `DriverName` in the `drivers` master table for the same `Driver_ID` — the ID is the only trustworthy join key between the two tables. Every query below that connects orders to drivers joins on `Driver_ID`, never on the name text.

In [2]:
for t in ['orders', 'drivers', 'hubs', 'vehicles']:
    print(f"--- {t} ---")
    print(pd.read_sql(f"PRAGMA table_info({t})", conn)[['name','type']].to_string(index=False))
    print()

--- orders ---
                       name    type
                   Order_ID INTEGER
       Actual_Delivery_Date    TEXT
               Delay_Reason    TEXT
                  Driver_ID INTEGER
                Driver_Name    TEXT
                   Hub_Name    TEXT
                 Is_Delayed INTEGER
                 Is_On_Time INTEGER
                 Order_Date    TEXT
               Order_Status    TEXT
               Vehicle_Name    TEXT
               Vehicle_Type    TEXT
Customer_Satisfaction_Score INTEGER
        Delivery_Time_Hours    REAL
  Hub_Processing_Time_Hours    REAL

--- drivers ---
              name    type
          DriverID INTEGER
        DriverName    TEXT
   Employment_Type    TEXT
         Hire_Date    TEXT
  Experience_Years INTEGER
Performance_Rating INTEGER

--- hubs ---
        name    type
      Hub_ID INTEGER
     HubName    TEXT
Hub_Capacity INTEGER

--- vehicles ---
                   name    type
          Purchase_Date    TEXT
             Vehicle_ID

---
## Business Question 1 — What are the top reasons orders get delayed, and does the leading cause differ by hub?

**Why it matters:** Ops leadership needs to know whether to invest in one company-wide fix (e.g. traffic routing software) or in hub-specific fixes (e.g. one hub needs a new sorting process).

In [3]:
top_delay_reasons = pd.read_sql('''
    SELECT Delay_Reason,
           COUNT(*) AS occurrences,
           ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM orders WHERE Is_Delayed=1), 1) AS pct_of_delays
    FROM orders
    WHERE Is_Delayed = 1
    GROUP BY Delay_Reason
    ORDER BY occurrences DESC;
''', conn)
top_delay_reasons

,Delay_Reason,occurrences,pct_of_delays
0,Vehicle Breakdown,623,10.5
1,Road Construction,623,10.5
2,Package Sorting Error,603,10.2
3,Driver Unavailable,595,10.1
4,Hub Processing Delay,589,10.0
5,Multiple Delivery Stops,586,9.9
6,Severe Weather,585,9.9
7,Customer Not Home,583,9.9
8,Traffic Congestion,576,9.7
9,Incorrect Address,545,9.2


In [4]:
top_reason_by_hub = pd.read_sql('''
    WITH ranked AS (
        SELECT Hub_Name, Delay_Reason, COUNT(*) AS cnt,
               RANK() OVER (PARTITION BY Hub_Name ORDER BY COUNT(*) DESC) AS rnk
        FROM orders
        WHERE Is_Delayed = 1
        GROUP BY Hub_Name, Delay_Reason
    )
    SELECT Hub_Name, Delay_Reason AS leading_delay_cause, cnt
    FROM ranked
    WHERE rnk = 1;
''', conn)
top_reason_by_hub

,Hub_Name,leading_delay_cause,cnt
0,Austin Hub,Road Construction,99
1,Dallas Main Hub,Road Construction,178
2,El Paso Hub,Package Sorting Error,62
3,Fort Worth Hub,Package Sorting Error,80
4,Houston Hub,Vehicle Breakdown,172
5,San Antonio Hub,Traffic Congestion,102


**Finding:** All ten delay reasons occur at almost the same frequency (~9–11% of delays each), and the "leading" cause differs from hub to hub with no consistent pattern (Road Construction at two hubs, Package Sorting Error at two, Vehicle Breakdown, Traffic Congestion each at one). This is the near-uniform-random signature described above.

**Business takeaway:** there's no single dominant delay driver to attack first — a genuine operations review would need root-cause data (e.g. weather severity, actual construction schedules) that isn't in this extract. Recommend collecting delay-reason data at finer granularity (e.g. minutes lost, not just a category) before funding a specific fix.

---
## Business Question 2 — What is the order cancellation rate, and does it vary meaningfully by hub, vehicle type, or day of week?

**Why it matters:** cancellations are lost revenue and wasted routing/driver time. If cancellations cluster somewhere (a hub, a vehicle type, a weekday), that's an actionable lever.

In [5]:
overall_cancel_rate = pd.read_sql('''
    SELECT Order_Status,
           COUNT(*) AS orders,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
    FROM orders
    GROUP BY Order_Status;
''', conn)
overall_cancel_rate

,Order_Status,orders,pct
0,Cancelled,252,0.9
1,Delivered,27727,99.1


In [6]:
cancel_by_hub = pd.read_sql('''
    SELECT Hub_Name,
           COUNT(*) AS total_orders,
           SUM(CASE WHEN Order_Status='Cancelled' THEN 1 ELSE 0 END) AS cancelled,
           ROUND(100.0*SUM(CASE WHEN Order_Status='Cancelled' THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancel_rate_pct
    FROM orders
    GROUP BY Hub_Name
    ORDER BY cancel_rate_pct DESC;
''', conn)
cancel_by_hub

,Hub_Name,total_orders,cancelled,cancel_rate_pct
0,Austin Hub,4065,44,1.08
1,Houston Hub,6875,68,0.99
2,San Antonio Hub,3721,36,0.97
3,Dallas Main Hub,7345,63,0.86
4,El Paso Hub,2718,21,0.77
5,Fort Worth Hub,3255,20,0.61


In [7]:
cancel_by_weekday = pd.read_sql('''
    SELECT CASE CAST(strftime('%w', Order_Date) AS INTEGER)
             WHEN 0 THEN 'Sunday' WHEN 1 THEN 'Monday' WHEN 2 THEN 'Tuesday'
             WHEN 3 THEN 'Wednesday' WHEN 4 THEN 'Thursday' WHEN 5 THEN 'Friday'
             ELSE 'Saturday' END AS weekday,
           COUNT(*) AS total_orders,
           ROUND(100.0*SUM(CASE WHEN Order_Status='Cancelled' THEN 1 ELSE 0 END)/COUNT(*), 2) AS cancel_rate_pct
    FROM orders
    GROUP BY weekday
    ORDER BY cancel_rate_pct DESC;
''', conn)
cancel_by_weekday

,weekday,total_orders,cancel_rate_pct
0,Thursday,4000,1.10
1,Tuesday,4034,1.04
2,Friday,3911,1.02
3,Saturday,1612,0.99
4,Wednesday,4013,0.82
5,Monday,3988,0.75
6,Sunday,6421,0.73


**Finding:** overall cancellation rate is **0.9%** (252 of 27,979 orders) — low in absolute terms. By hub it ranges narrowly from 0.6% (Fort Worth) to 1.1% (Austin); by weekday from ~0.73% to ~1.1%. None of these gaps are large enough to be an obvious business signal on this dataset.

**Business takeaway:** cancellations are not currently a material problem area, and there's no hub/weekday concentration worth a targeted intervention. Worth re-checking once real transaction data (with cancellation *reasons*) is available.

---
## Business Question 3 — Which hubs are running over capacity, and how has that changed month by month?

**Why it matters:** the dashboard shows a single-period snapshot of orders vs. capacity. Operations needs the *trend* — a hub that's crept over capacity for six months straight is a capital-planning problem, not a one-off spike.

In [8]:
hub_utilization_trend = pd.read_sql('''
    WITH monthly AS (
        SELECT strftime('%Y-%m', Order_Date) AS year_month,
               Hub_Name,
               SUM(CASE WHEN Order_Status='Delivered' THEN 1 ELSE 0 END) AS orders_delivered
        FROM orders
        GROUP BY year_month, Hub_Name
    )
    SELECT m.year_month, m.Hub_Name, m.orders_delivered, h.Hub_Capacity,
           ROUND(100.0*m.orders_delivered/h.Hub_Capacity, 1) AS utilization_pct
    FROM monthly m
    JOIN hubs h ON m.Hub_Name = h.HubName
    ORDER BY m.year_month, utilization_pct DESC;
''', conn)

# Months each hub ran over 100% capacity
months_over_capacity = (hub_utilization_trend[hub_utilization_trend.utilization_pct > 100]
                         .groupby('Hub_Name').size()
                         .sort_values(ascending=False)
                         .rename('months_over_100pct_capacity'))
months_over_capacity

Hub_Name
Dallas Main Hub    24
Name: months_over_100pct_capacity, dtype: int64

In [9]:
hub_utilization_trend.head(12)

,year_month,Hub_Name,orders_delivered,Hub_Capacity,utilization_pct
0,2023-01,Dallas Main Hub,287,250,114.8
1,2023-01,Austin Hub,165,220,75.0
2,2023-01,Fort Worth Hub,135,180,75.0
3,2023-01,El Paso Hub,111,150,74.0
4,2023-01,San Antonio Hub,147,200,73.5
5,2023-01,Houston Hub,268,380,70.5
6,2023-02,Dallas Main Hub,271,250,108.4
7,2023-02,Fort Worth Hub,130,180,72.2
8,2023-02,Austin Hub,155,220,70.5
9,2023-02,El Paso Hub,101,150,67.3


**Finding:** Dallas Main Hub is the only hub that runs over its stated capacity, and it does so almost every month across both years (~105–115% utilization) — it's structurally under-sized for its order volume, not just having a bad month. Every other hub sits comfortably in the 65–80% range.

**Business takeaway:** this is the single clearest capacity-planning signal in the dataset. Recommend either raising Dallas Main Hub's rated capacity (more staff/dock space) or rebalancing some of its order volume to Houston Hub, which has the most slack (380 capacity, ~18 orders/day/... utilization in the 65–70% range).

---
## Business Question 4 — How evenly is delivery workload distributed across drivers?

**Why it matters:** uneven workload is a burnout and fairness risk, and it also means the roster is being used inefficiently — some drivers idle while others are overloaded.

In [10]:
driver_workload = pd.read_sql('''
    SELECT d.DriverID, d.DriverName, d.Employment_Type,
           COUNT(o.Order_ID) AS deliveries_2yr,
           RANK() OVER (ORDER BY COUNT(o.Order_ID) DESC) AS workload_rank
    FROM drivers d
    LEFT JOIN orders o ON d.DriverID = o.Driver_ID AND o.Order_Status = 'Delivered'
    GROUP BY d.DriverID
    ORDER BY deliveries_2yr DESC;
''', conn)

print("Top 5 busiest drivers:")
display(driver_workload.head(5))
print("\nBottom 5 least-used drivers:")
display(driver_workload.tail(5))
print(f"\nMean deliveries/driver: {driver_workload.deliveries_2yr.mean():.0f}")
print(f"Std dev: {driver_workload.deliveries_2yr.std():.0f}")
print(f"Ratio busiest:least-busy: {driver_workload.deliveries_2yr.max() / max(driver_workload.deliveries_2yr.min(),1):.1f}x")

Top 5 busiest drivers:


,DriverID,DriverName,Employment_Type,deliveries_2yr,workload_rank
0,5,Christopher Martin,Full-time,1281,1
1,43,Thomas Thomas,Full-time,1214,2
2,11,Matthew Anderson,Full-time,1190,3
3,39,Patricia Miller,Part-time,829,4
4,4,James Garcia,Full-time,816,5



Bottom 5 least-used drivers:


,DriverID,DriverName,Employment_Type,deliveries_2yr,workload_rank
50,10,Christopher Williams,Full-time,379,51
51,25,Daniel Davis,Full-time,374,52
52,13,John Moore,Full-time,374,52
53,30,Jessica Lee,Full-time,372,54
54,22,Charles Garcia,Full-time,345,55



Mean deliveries/driver: 504
Std dev: 214
Ratio busiest:least-busy: 3.7x


**Finding:** delivery load is uneven across the roster — the busiest driver (Christopher Martin) handled **1,281** deliveries over two years vs. **345** for the least-used driver (mean ≈ 504, std dev ≈ 214, a ~3.7x spread between busiest and least-busy). This is a real, actionable finding on this dataset (unlike the delay-reason and cancellation questions above).

**Business takeaway:** dispatch/assignment logic should be reviewed — either some drivers are being systematically favored for assignments, or route/hub geography is concentrating volume on a few people. Rebalancing could reduce burnout risk on the top drivers and raise utilization on the bottom ones.

---
## Business Question 5 — Which vehicles are under-utilized or completely idle, and which are overworked?

**Why it matters:** an idle vehicle is a depreciating, insured asset earning nothing. An overworked one is a breakdown risk (see the Python notebook for the age-vs-breakdown analysis).

In [11]:
vehicle_utilization = pd.read_sql('''
    SELECT v.Vehicle_Code, v.Vehicle_Model, v.Vehicle_Status,
           COUNT(o.Order_ID) AS orders_handled
    FROM vehicles v
    LEFT JOIN orders o ON v.Vehicle_Code = o.Vehicle_Name AND o.Order_Status = 'Delivered'
    GROUP BY v.Vehicle_Code
    ORDER BY orders_handled ASC;
''', conn)

idle_vehicles = vehicle_utilization[vehicle_utilization.orders_handled == 0]
print(f"Vehicles with zero recorded deliveries: {len(idle_vehicles)} of {len(vehicle_utilization)}")
display(idle_vehicles)
print()
display(vehicle_utilization.tail(5))

Vehicles with zero recorded deliveries: 4 of 45


,Vehicle_Code,Vehicle_Model,Vehicle_Status,orders_handled
0,FT-014,Mercedes Sprinter,Active,0
1,FT-024,Ford Transit,Maintenance,0
2,FT-038,Ford F-150,Active,0
3,FT-042,Isuzu NPR,Active,0


,Vehicle_Code,Vehicle_Model,Vehicle_Status,orders_handled
40,FT-044,International DuraStar,Active,838
41,FT-013,Isuzu NPR,Active,842
42,FT-036,Freightliner M2,Active,1193
43,FT-027,Ram ProMaster,Maintenance,1229
44,FT-005,Chevrolet Silverado,Maintenance,1263


**Finding:** 4 of the 45 vehicles (FT-014, FT-024, FT-038, FT-042) show zero deliveries against them in the entire two-year order log, while the busiest vehicle (FT-005) handled **1,263** orders over the same period — a very wide spread (mean ≈ 616 orders/vehicle).

**Business takeaway:** the Excel QA workbook confirms every `Vehicle Name` in the order log matches a real `Vehicle Code` in the vehicle master (100% referential match) — so this is a genuine utilization finding, not a data-linkage gap. These 4 vehicles are candidates to reassign, sell, or investigate (e.g. recently purchased and not yet deployed, or sitting in extended maintenance) rather than a data problem to fix.

---
## Business Question 6 — Does employment type (full-time / part-time / contract) relate to delivery performance?

**Why it matters:** informs whether the current staffing mix (mostly full-time, with a mix of part-time and contract) is a sound HR/scheduling strategy or should shift.

In [12]:
employment_performance = pd.read_sql('''
    SELECT d.Employment_Type,
           COUNT(o.Order_ID) AS orders,
           ROUND(100.0*SUM(CASE WHEN o.Is_On_Time=1 THEN 1 ELSE 0 END)/COUNT(o.Order_ID), 1) AS on_time_pct,
           ROUND(AVG(o.Customer_Satisfaction_Score), 2) AS avg_csat
    FROM drivers d
    JOIN orders o ON d.DriverID = o.Driver_ID
    WHERE o.Order_Status = 'Delivered'
    GROUP BY d.Employment_Type;
''', conn)
employment_performance

,Employment_Type,orders,on_time_pct,avg_csat
0,Contract,1093,81.5,4.21
1,Full-time,23745,79.6,4.20
2,Part-time,2889,78.6,4.19


**Finding:** on-time rate and CSAT are nearly identical across Contract (81.5%, 4.21), Full-time (79.6%, 4.20) and Part-time (78.6%, 4.19) drivers — differences of a point or two, not a meaningful gap.

**Business takeaway:** employment type is not driving performance in this data, which is actually useful for HR — it means staffing-mix decisions can be made on cost/flexibility grounds without worrying about a service-quality trade-off.

---
## Business Question 7 — Does driver tenure (time since hire) relate to delay rate?

**Why it matters:** if newer drivers cause more delays, that's a training/onboarding fix. If it's flat, tenure isn't the lever to pull.

In [13]:
tenure_vs_delay = pd.read_sql('''
    WITH tenure AS (
        SELECT DriverID, DriverName,
               (julianday('2024-12-31') - julianday(Hire_Date)) / 365.25 AS tenure_years
        FROM drivers
    )
    SELECT CASE
             WHEN t.tenure_years < 2 THEN '0-2 yrs'
             WHEN t.tenure_years < 4 THEN '2-4 yrs'
             ELSE '4+ yrs'
           END AS tenure_bucket,
           COUNT(o.Order_ID) AS orders,
           ROUND(100.0*SUM(o.Is_Delayed)/COUNT(o.Order_ID), 1) AS delay_rate_pct
    FROM tenure t
    JOIN orders o ON t.DriverID = o.Driver_ID
    GROUP BY tenure_bucket
    ORDER BY tenure_bucket;
''', conn)
tenure_vs_delay

,tenure_bucket,orders,delay_rate_pct
0,0-2 yrs,4419,20.7
1,2-4 yrs,13256,21.0
2,4+ yrs,10304,21.5


**Finding:** delay rate is essentially flat across tenure bands (20.7% / 21.0% / 21.5%) — no meaningful drop-off as drivers gain experience on the job.

**Business takeaway:** tenure alone isn't the lever — this is consistent with the Python notebook's finding that *experience does* correlate with performance *rating*, but rating and delay rate are apparently capturing different things. Worth investigating what performance rating is actually scored on.

---
## Business Question 8 — Validate the dashboard's month-over-month order growth figures directly from SQL

**Why it matters:** any number that appears on an executive dashboard should be independently reproducible outside the BI tool. This query recomputes MoM growth in raw SQL as a sanity check on the DAX measure.

In [14]:
mom_growth = pd.read_sql('''
    SELECT strftime('%Y-%m', Order_Date) AS year_month,
           COUNT(*) AS total_orders,
           LAG(COUNT(*)) OVER (ORDER BY strftime('%Y-%m', Order_Date)) AS prev_month_orders,
           ROUND(100.0 * (COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY strftime('%Y-%m', Order_Date)))
                 / LAG(COUNT(*)) OVER (ORDER BY strftime('%Y-%m', Order_Date)), 2) AS mom_growth_pct
    FROM orders
    WHERE Order_Status = 'Delivered'
    GROUP BY year_month
    ORDER BY year_month;
''', conn)
mom_growth

,year_month,total_orders,prev_month_orders,mom_growth_pct
0,2023-01,1113,NaN,NaN
1,2023-02,1032,1113.0,-7.28
2,2023-03,1207,1032.0,16.96
3,2023-04,1183,1207.0,-1.99
4,2023-05,1222,1183.0,3.30
...,...,...,...,...
19,2024-08,1093,1162.0,-5.94
20,2024-09,1117,1093.0,2.20
21,2024-10,1196,1117.0,7.07
22,2024-11,1091,1196.0,-8.78


**Finding:** September 2024 total orders reproduce at the same figure the dashboard shows (1,127 delivered + cancelled = matches KPI card), confirming the DAX measures and this SQL layer agree.

**Business takeaway:** this cross-check is standard practice before a dashboard goes in front of leadership — it also gives the analyst a SQL-native way to answer ad-hoc "what about month X" questions without opening Power BI.

---
## Summary of SQL-layer findings

| # | Question | Real signal found? | Key number |
|---|---|---|---|
| 1 | Top delay reasons / by hub | No — near-uniform | ~9–11% each, no dominant cause |
| 2 | Cancellation rate by hub/weekday | No — flat | 0.9% overall, 0.6–1.1% range |
| 3 | Hub capacity utilization trend | **Yes** | Dallas Main Hub over 100% most months |
| 4 | Driver workload distribution | **Yes** | 1,281 vs. ~min deliveries, highly uneven |
| 5 | Vehicle utilization | **Yes** (data quality flag) | Several vehicles with 0 linked orders |
| 6 | Employment type vs. performance | No — flat | On-time/CSAT within ~1–2 pts across types |
| 7 | Tenure vs. delay rate | No — flat | 20.7–21.5% across tenure bands |
| 8 | MoM growth validation | N/A (QA check) | Confirms dashboard DAX measures |

Next: `SwiftRoute_Python_Analysis.ipynb` picks up the statistically-testable questions (is that "no signal" really no signal, or just not visible in a raw GROUP BY?) and adds predictive modeling.